# Fetch NBA Player Game Logs

Pulls all player game logs from `nba_api` for each configured season and saves them to `data/raw/`.

Each season is saved as `game_logs_<season>.csv`. Seasons already on disk are skipped automatically.

In [ ]:
import sys
from pathlib import Path

# Make sure project root is on the path so src.data.fetch is importable
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import yaml

cfg = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text())

SEASONS   = cfg["data"]["seasons"]
RAW_DIR   = ROOT / cfg["data"]["raw_dir"]
API_DELAY = 0.6  # seconds between requests — stays within nba_api rate limit

print("Seasons to fetch:", SEASONS)
print("Output directory:", RAW_DIR)

In [ ]:
from src.data.fetch import fetch_all_seasons

paths = fetch_all_seasons(SEASONS, output_dir=RAW_DIR, delay=API_DELAY)

print("\nFiles written:")
for p in paths:
    print(f"  {p}")

In [ ]:
import pandas as pd

frames = []
for p in paths:
    df = pd.read_csv(p)
    df["season"] = p.stem.replace("game_logs_", "").replace("_", "-")
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)

print(f"Total rows: {len(combined):,}")
print(f"Unique players: {combined['PLAYER_ID'].nunique():,}")
print(f"Seasons: {sorted(combined['season'].unique())}")
combined.head()

In [ ]:
print("Column list:")
for col in combined.columns:
    print(f"  {col}")

In [ ]:
# Quick sanity-check on target stats
target_cols = [c.upper() for c in cfg["features"]["target_stats"]]
print("Target stat summaries:")
combined[target_cols].describe().round(2)